# Semantic Chunking

Split where the *meaning* changes: a breakpoint appears when neighbouring sentences are dissimilar.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *

text = load_text()
sentences = split_sentences(text.replace("## ", "").replace("# ", ""))
vecs = embed(sentences)
print(len(sentences), "sentences")

Distance between sentence *i* and *i+1* is `1 - cosine`. Spikes above a percentile threshold mark topic changes.

In [ ]:
import numpy as np

dist = 1 - np.sum(vecs[:-1] * vecs[1:], axis=1)
threshold = np.percentile(dist, 85)
breaks = [i + 1 for i, d in enumerate(dist) if d > threshold]

semantic_chunks, start = [], 0
for b in breaks + [len(sentences)]:
    semantic_chunks.append(" ".join(sentences[start:b]))
    start = b
print(f"threshold={threshold:.3f}  ->  {len(semantic_chunks)} semantic chunks")
print(semantic_chunks[3][:300])

In [ ]:
sem_store = build_store(semantic_chunks)
query = "What is the all-MiniLM-L6-v2 model good for?"
response, hits = rag(query, sem_store, k=2)
show(hits); print(response)